In [1]:
from cryptography.hazmat.primitives.ciphers.aead import ChaCha20Poly1305
import os

In [2]:
key = ChaCha20Poly1305.generate_key()

print("Key length:", len(key))

Key length: 32


In [3]:
cipher = ChaCha20Poly1305(key)

In [4]:
payload = b"""
{   "command": "takeoff",\
    "target": "dZONE_A",\
    "speed": 5\
}
"""

print("Payload length:", len(payload))

Payload length: 66


In [12]:
nonce = os.urandom(12)

print("Nonce length:", len(nonce))
print("Nonce:", nonce.hex())

Nonce length: 12
Nonce: db00a0a7701e0f7a810c82af


In [13]:
aad = b"command_metadata"

ciphertext = cipher.encrypt(nonce, payload, aad)

print("Ciphertext length:", len(ciphertext))

Ciphertext length: 82


In [16]:
decrypted_payload = cipher.decrypt(nonce, ciphertext, aad)
print("Decrypted payload:", decrypted_payload)

Decrypted payload: b'\n{   "command": "takeoff",    "target": "dZONE_A",    "speed": 5}\n'


In [17]:
tampered = bytearray(ciphertext)
tampered[5] ^= 1
tampered = bytes(tampered)

In [18]:
try:
    cipher.decrypt(nonce, tampered, aad)
    print("Unexpected success")
except Exception as e:
    print("Tampering detected → authentication failed")

Tampering detected → authentication failed


In [19]:
try:
    cipher.decrypt(nonce, ciphertext, b"random_aad")
    print("Unexpected success")
except Exception as e:
    print("AAD mismatch detected → authentication failed")

AAD mismatch detected → authentication failed
